# Kenya Monthly Climate & Malaria Resistance Dashboard — Interactive Demo Notebook

This notebook provides a complete interactive demonstration of all maps, visualizations, demographic analyses, and ecological statistical/machine learning models featured on the **Kenya Climate & Vegetation Dashboard**.

### Notebook Contents:
1. **Environment Setup & Data Ingestion**: Direct ingestion from 5 km GeoTIFF multi-band rasters (`kenya_YYYY_MM.tif`) and parquet cache (`combined_part*.parquet`), county boundaries (`counties.geojson`), and 100,000 simulated malaria patient records.
2. **5 km High-Resolution Grid Map**: Interactive spatial visualization of climate metrics.
3. **County Choropleth Climate Map**: Spatial aggregation across Kenya's 47 counties.
4. **Simulated Malaria Resistance & Demographics**: County-level HbAS allele prevalence map, sex/age distributions, and climate correlations.
5. **Monthly Trend Analysis**: National time-series and multi-county climate trends.
6. **Ecological Statistical & ML Models**: LASSO Binomial GLM, Random Forest, and XGBoost with 5-fold cross-validation, hyperparameter tuning, and feature importances.
7. **Model Performance Summary**: Comparative metrics table ($R^2$, MSE).

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import json
import pathlib
import statsmodels.api as sm
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import KFold, GridSearchCV
try:
    from IPython.display import display
except ImportError:
    display = print

from dashboard_data import (
    PARQUET_CACHE,
    VARIABLES,
    TIFF_BAND_NAMES,
    read_tiff_file,
    load_combined,
    county_monthly_averages
)

print('[OK] Environment initialized and modules imported successfully!')

## 1. GeoTIFF Raster Data Ingestion & Preprocessing
We load the primary datasets used by the dashboard:
- **`kenya_YYYY_MM.tif`**: Multi-band 5 km GeoTIFF rasters containing 18 climate and land cover bands.
- **`combined_part*.parquet`**: Consolidated high-resolution monthly grid compiled directly from GeoTIFF rasters.
- **`counties.geojson`**: Administrative shape boundaries for Kenya's 47 counties.
- **`simulated_malaria_patients.csv`**: Synthetic dataset of 100,000 patients with demographics, genotypes (HbAA vs HbAS sickle trait), and location coordinates.

In [ ]:
# 1. Demonstrate reading a single GeoTIFF raster (.tif) using rasterio
sample_tif = pathlib.Path("kenya_monthly_ingest/kenya_2026_02.tif")
if sample_tif.exists():
    single_tif_df = read_tiff_file(sample_tif)
    print(f"Read single GeoTIFF '{sample_tif.name}': {len(single_tif_df):,} pixels across {len(TIFF_BAND_NAMES)} climate bands.")

# 2. Load full combined dataset (compiled from GeoTIFF rasters)
df = load_combined()
print(f"Loaded full climate grid: {len(df):,} pixel-months across {df['month'].nunique()} months ({df['month'].min().strftime('%Y-%m')} to {df['month'].max().strftime('%Y-%m')}).")

# 3. Load county boundaries GeoJSON
geojson_path = pathlib.Path("kenya_monthly_ingest/counties.geojson")
with open(geojson_path, "r") as f:
    counties_geojson = json.load(f)
print(f"Loaded GeoJSON with {len(counties_geojson['features'])} county features.")

# 4. Compute county monthly averages
county_df = county_monthly_averages(df)
print(f"Computed county monthly averages: {len(county_df)} records across {county_df['county'].nunique()} counties.")

# 5. Load simulated patient dataset
patient_file = pathlib.Path("simulated_malaria_patients.csv")
patient_df = pd.read_csv(patient_file)
print(f"Loaded simulated patient database: {len(patient_df):,} patient records.")

## 2. 5 km High-Resolution Spatial Grid Maps
The dashboard renders 5 km grid point scatter maps across Kenya. Below is an interactive map displaying **NDVI (Normalized Difference Vegetation Index)** for the most recent month in the dataset.

In [ ]:
latest_month = sorted(df["month"].unique())[-1]
month_df = df[df["month"] == latest_month].dropna(subset=["ndvi"])

fig_grid = px.scatter_map(
    month_df,
    lat="lat",
    lon="lon",
    color="ndvi",
    color_continuous_scale="YlGn",
    center=dict(lat=0.5, lon=37.9),
    zoom=5.2,
    map_style="open-street-map",
    height=650,
    title=f"Kenya 5 km Grid NDVI — {latest_month.strftime('%Y-%m')}",
    labels={"ndvi": "NDVI (Vegetation Greenness)"}
)
fig_grid.update_traces(marker=dict(size=6, opacity=0.8))
fig_grid.update_layout(margin=dict(l=0, r=0, t=40, b=0))
fig_grid.show()

## 3. County-Level Climate Choropleth Maps
Aggregating climate variables to county administrative boundaries allows spatial comparison across Kenya's 47 counties. Below is an interactive choropleth map of **Monthly Rainfall (mm)**.

In [ ]:
county_latest = county_df[county_df["month"] == latest_month].dropna(subset=["rain_mm"])

fig_county_rain = px.choropleth_map(
    county_latest,
    geojson=counties_geojson,
    locations="county",
    featureidkey="properties.shapeName",
    color="rain_mm",
    color_continuous_scale="Blues",
    center=dict(lat=0.5, lon=37.9),
    zoom=5.2,
    map_style="open-street-map",
    height=650,
    title=f"Kenya County Monthly Rainfall (mm) — {latest_month.strftime('%Y-%m')}",
    labels={"rain_mm": "Rainfall (mm)"},
    opacity=0.75
)
fig_county_rain.update_layout(margin=dict(l=0, r=0, t=40, b=0))
fig_county_rain.show()

## 4. Simulated Malaria Resistance & Demographic Analysis
Here we analyze the spatial prevalence of the **HbAS (Sickle Cell Trait)** allele, which provides partial resistance against severe *Plasmodium falciparum* malaria.

Visualizations include:
1. **HbAS Prevalence Choropleth Map** by Kenya County.
2. **Genotype Distribution by Sex**.
3. **Prevalence Rate by Age Group**.
4. **Climate vs HbAS Prevalence Scatter Plot** (Temperature, Rainfall, Elevation).

In [ ]:
# Calculate HbAS mutation indicator & prevalence
patient_df['is_mutant'] = patient_df['human_genotype'] == 'HbAS (Sickle Trait / Resistant)'
res_cases = patient_df['is_mutant'].sum()
res_rate = (res_cases / len(patient_df)) * 100
print(f"Overall HbAS Sickle Trait Prevalence: {res_rate:.2f}% ({res_cases:,} / {len(patient_df):,} patients)")

county_res = patient_df.groupby('county')['is_mutant'].mean().reset_index()
county_res['Prevalence Rate (%)'] = county_res['is_mutant'] * 100

# Map 3: Malaria Resistance Choropleth
fig_res_map = px.choropleth_map(
    county_res,
    geojson=counties_geojson,
    locations="county",
    featureidkey="properties.shapeName",
    color="Prevalence Rate (%)",
    color_continuous_scale="Reds",
    center=dict(lat=0.5, lon=37.9),
    zoom=5.2,
    map_style="open-street-map",
    height=650,
    title="HbAS (Sickle Trait) Malaria Resistance Prevalence by Kenya County",
    opacity=0.75
)
fig_res_map.update_layout(margin=dict(l=0, r=0, t=40, b=0))
fig_res_map.show()

# Demographics Plot 1: Genotypes by Sex
sex_res = patient_df.groupby(['sex', 'human_genotype']).size().reset_index(name='count')
fig_sex = px.bar(sex_res, x='sex', y='count', color='human_genotype', barmode='group', title="Genotype Distribution by Sex")
fig_sex.show()

# Demographics Plot 2: HbAS Prevalence by Age Group
patient_df['Age Group'] = pd.cut(patient_df['age'], bins=[0, 10, 20, 30, 40, 50, 60, 100], labels=['0-10', '11-20', '21-30', '31-40', '41-50', '51-60', '60+'])
age_res = patient_df.groupby('Age Group', observed=False)['is_mutant'].mean().reset_index()
age_res['Rate (%)'] = age_res['is_mutant'] * 100
fig_age = px.line(age_res, x='Age Group', y='Rate (%)', title="HbAS Trait Rate by Age Group", markers=True)
fig_age.show()

# Demographics Plot 3: Climate Correlations
baseline_climate = county_df.groupby('county')[['mean_temp_c', 'rain_mm', 'elevation_m']].mean().reset_index()
res_climate = county_res.merge(baseline_climate, on='county')
fig_scatter = px.scatter(
    res_climate,
    x='mean_temp_c',
    y='Prevalence Rate (%)',
    color='rain_mm',
    size='elevation_m',
    hover_name='county',
    title="Mean Temperature, Rainfall & Elevation vs HbAS Prevalence by County",
    labels={'mean_temp_c': 'Mean Temperature (°C)', 'rain_mm': 'Rainfall (mm)'}
)
fig_scatter.show()

## 5. Monthly Climate Trend Analysis
We inspect monthly temporal trends over time across the 25+ year dataset (2001–2026).
1. **National Mean NDVI Trend**.
2. **County Mean Temperature Comparison** across representative regions.

In [ ]:
# National NDVI trend over time
national_trend = df.groupby("month")["ndvi"].mean().reset_index()
fig_trend = px.line(
    national_trend,
    x="month",
    y="ndvi",
    title="Kenya National Mean NDVI Trend (2001 – 2026)",
    labels={"ndvi": "Mean NDVI", "month": "Date"}
)
fig_trend.show()

# Multi-county comparison for Temperature
sample_counties = ["Nairobi", "Mombasa", "Kisumu", "Garissa", "Turkana"]
county_sample_df = county_df[county_df["county"].isin(sample_counties)]
fig_county_comp = px.line(
    county_sample_df,
    x="month",
    y="mean_temp_c",
    color="county",
    title="Monthly Mean Temperature (°C) Comparison Across Select Counties",
    labels={"mean_temp_c": "Mean Temp (°C)", "month": "Date"}
)
fig_county_comp.show()

## 6. Ecological Machine Learning & Statistical Modeling
The dashboard includes a statistical modeling engine to evaluate associations between standardized climatic variables (plus 2nd-degree polynomial interaction terms) and county-level malaria resistance prevalence.

We train and evaluate three distinct models:
1. **LASSO Regression (L1 Penalty Binomial GLM)**: ElasticNet GLM with 5-fold cross-validation.
2. **Random Forest Regressor**: Non-linear ensemble model tuned with 5-fold `GridSearchCV`.
3. **XGBoost Regressor**: Gradient boosting model tuned with 5-fold `GridSearchCV`.

In [ ]:
# Aggregate patient target variable per county
county_table = patient_df.groupby('county').agg(
    n_resistant=('is_mutant', 'sum'),
    n_tested=('patient_id', 'count')
).reset_index()
county_table['n_susceptible'] = county_table['n_tested'] - county_table['n_resistant']

# Baseline climate predictors
predictors = ['mean_temp_c', 'max_temp_c', 'min_temp_c', 'rain_mm', 'humidity_rh_pct', 'soil_moisture_m3m3', 'wind_u', 'wind_v', 'ndvi', 'elevation_m', 'urban_pct']
available_predictors = [p for p in predictors if p in county_df.columns]

baseline_df = county_df.groupby('county')[available_predictors].mean().reset_index()
model_df = county_table.merge(baseline_df, on='county')

# Standardize predictors (Z-scores)
for p in available_predictors:
    model_df[f'z_{p}'] = (model_df[p] - model_df[p].mean()) / model_df[p].std()

# Target & Feature matrix with 2nd-degree polynomial interactions
y = model_df['n_resistant'] / model_df['n_tested']
X_base = model_df[[f'z_{p}' for p in available_predictors]]

poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_poly = poly.fit_transform(X_base)
feature_names = poly.get_feature_names_out(X_base.columns)
X = pd.DataFrame(X_poly, columns=feature_names)

print(f"Modeling dataset ready: {len(model_df)} counties, {X.shape[1]} interaction features.")

In [ ]:
print("Training Model 1: LASSO Binomial GLM with 5-Fold Cross-Validation...")
endog = model_df[['n_resistant', 'n_susceptible']]
exog = sm.add_constant(X)

alphas = [0.0001, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5]
kf = KFold(n_splits=5, shuffle=True, random_state=42)
best_alpha = alphas[0]
best_lasso_mse = float('inf')

for alpha in alphas:
    fold_mses = []
    for train_idx, test_idx in kf.split(X):
        train_endog, test_endog = endog.iloc[train_idx], endog.iloc[test_idx]
        train_exog, test_exog = exog.iloc[train_idx], exog.iloc[test_idx]
        try:
            glm = sm.GLM(train_endog, train_exog, family=sm.families.Binomial())
            res = glm.fit_regularized(method='elastic_net', alpha=alpha, L1_wt=1.0)
            y_test_pred = res.predict(test_exog)
            y_test_true = test_endog['n_resistant'] / (test_endog['n_resistant'] + test_endog['n_susceptible'])
            fold_mses.append(mean_squared_error(y_test_true, y_test_pred))
        except Exception:
            fold_mses.append(float('inf'))
    avg_mse = np.mean(fold_mses)
    if avg_mse < best_lasso_mse:
        best_lasso_mse = avg_mse
        best_alpha = alpha

glm = sm.GLM(endog, exog, family=sm.families.Binomial())
lasso_res = glm.fit_regularized(method='elastic_net', alpha=best_alpha, L1_wt=1.0)
y_lasso_pred = lasso_res.predict(exog)

lasso_r2 = r2_score(y, y_lasso_pred)
lasso_mse = mean_squared_error(y, y_lasso_pred)
print(f"Optimal Alpha: {best_alpha}")
print(f"LASSO GLM -> R2: {lasso_r2:.4f}, MSE: {lasso_mse:.6f}")

# Plot Top 20 LASSO Coefficients
lasso_coefs = lasso_res.params.drop('const', errors='ignore').values
lasso_imp_df = pd.DataFrame({'Predictor': feature_names, 'Importance': lasso_coefs})
lasso_imp_df['Abs_Importance'] = lasso_imp_df['Importance'].abs()
top_lasso = lasso_imp_df.sort_values(by='Abs_Importance', ascending=False).head(20).sort_values(by='Abs_Importance', ascending=True)

fig_lasso = px.bar(
    top_lasso, x='Importance', y='Predictor', orientation='h',
    title=f"LASSO Regression Coefficients (Top 20, Optimal Alpha={best_alpha})",
    color='Importance', color_continuous_scale="RdBu"
)
fig_lasso.add_vline(x=0.0, line_width=2, line_color="black")
fig_lasso.show()

In [ ]:
print("Training Model 2: Random Forest Regressor with 5-Fold GridSearchCV...")
rf_base = RandomForestRegressor(random_state=42)
rf_param_grid = {'n_estimators': [50, 100, 200], 'max_depth': [None, 3, 5]}
rf_grid = GridSearchCV(rf_base, rf_param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
rf_grid.fit(X, y)

rf_best = rf_grid.best_estimator_
y_rf_pred = rf_best.predict(X)
rf_r2 = r2_score(y, y_rf_pred)
rf_mse = mean_squared_error(y, y_rf_pred)

print(f"Best Parameters: {rf_grid.best_params_}")
print(f"Random Forest -> R2: {rf_r2:.4f}, MSE: {rf_mse:.6f}")

# Feature Importance Plot
rf_imp_df = pd.DataFrame({'Predictor': feature_names, 'Importance': rf_best.feature_importances_})
rf_imp_df['Abs_Importance'] = rf_imp_df['Importance'].abs()
top_rf = rf_imp_df.sort_values(by='Abs_Importance', ascending=False).head(20).sort_values(by='Abs_Importance', ascending=True)

fig_rf = px.bar(
    top_rf, x='Importance', y='Predictor', orientation='h',
    title="Random Forest Feature Importance (Top 20)",
    color='Importance', color_continuous_scale="Viridis"
)
fig_rf.show()

In [ ]:
print("Training Model 3: XGBoost Regressor with 5-Fold GridSearchCV...")
xgb_base = xgb.XGBRegressor(random_state=42, objective='reg:squarederror')
xgb_param_grid = {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.05, 0.1], 'max_depth': [3, 5]}
xgb_grid = GridSearchCV(xgb_base, xgb_param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
xgb_grid.fit(X, y)

xgb_best = xgb_grid.best_estimator_
y_xgb_pred = xgb_best.predict(X)
xgb_r2 = r2_score(y, y_xgb_pred)
xgb_mse = mean_squared_error(y, y_xgb_pred)

print(f"Best Parameters: {xgb_grid.best_params_}")
print(f"XGBoost -> R2: {xgb_r2:.4f}, MSE: {xgb_mse:.6f}")

# Feature Importance Plot
xgb_imp_df = pd.DataFrame({'Predictor': feature_names, 'Importance': xgb_best.feature_importances_})
xgb_imp_df['Abs_Importance'] = xgb_imp_df['Importance'].abs()
top_xgb = xgb_imp_df.sort_values(by='Abs_Importance', ascending=False).head(20).sort_values(by='Abs_Importance', ascending=True)

fig_xgb = px.bar(
    top_xgb, x='Importance', y='Predictor', orientation='h',
    title="XGBoost Relative Feature Importance (Top 20)",
    color='Importance', color_continuous_scale="Viridis"
)
fig_xgb.show()

## 7. Model Performance Comparison Summary
Below is a comparison of all three ecological machine learning models evaluated on the dashboard.

In [ ]:
metrics_summary = pd.DataFrame([
    {"Model": "LASSO Regression (Binomial GLM)", "Optimal Hyperparameters": f"alpha={best_alpha}", "R-squared (R2)": round(lasso_r2, 4), "Mean Squared Error (MSE)": round(lasso_mse, 6)},
    {"Model": "Random Forest Regressor", "Optimal Hyperparameters": str(rf_grid.best_params_), "R-squared (R2)": round(rf_r2, 4), "Mean Squared Error (MSE)": round(rf_mse, 6)},
    {"Model": "XGBoost Regressor", "Optimal Hyperparameters": str(xgb_grid.best_params_), "R-squared (R2)": round(xgb_r2, 4), "Mean Squared Error (MSE)": round(xgb_mse, 6)}
])

print("=== Dashboard Machine Learning Model Comparison ===")
display(metrics_summary)